# Dasar LangChain: Model, Pesan, Prompt, dan Structured Output: Solusi Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari3/01-dasar-langchain-solusi.ipynb)

Ini **solusi lengkap** semua latihan bab ini (3 latihan) di Google Colab. Semua bagian `TODO` sudah terisi, jadi notebook ini tinggal dijalankan (**Runtime > Run all**, atau `Shift+Enter` per cell). Keluaran dari satu kali eksekusi dan penjelasan tiap latihan sudah disertakan; jawaban model berubah tiap kali dijalankan, jadi hasil Anda bisa sedikit berbeda. Kerjakan dulu [notebook latihannya](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari3/01-dasar-langchain-latihan.ipynb); buka solusi ini untuk mencocokkan hasil Anda atau saat macet.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Bab 1 - Solusi Latihan 1: Chatbot Serah Terima Shift
Jalankan sel satu per satu dengan Shift+Enter

Chatbot dengan riwayat percakapan: setiap giliran ditambahkan ke daftar pesan lalu seluruhnya dikirim ke model.

Bagian utama memakai dua giliran contoh. Sel terakhir membuka mode interaktif (ketik catatan sendiri)
setelah `INTERAKTIF` diubah menjadi `True`.

In [ ]:
# Langkah 1 - Menyiapkan model dan riwayat percakapan
import os

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

messages = [
    SystemMessage(
        content="Anda adalah asisten serah terima shift untuk operator produksi migas. "
                "Catat informasi yang disampaikan operator, ingat semuanya sepanjang percakapan, "
                "dan jawab ringkas dalam Bahasa Indonesia."
    ),
]

print("Asisten: Halo, saya asisten serah terima shift. Sampaikan catatan Anda.")

Asisten: Halo, saya asisten serah terima shift. Sampaikan catatan Anda.


## Langkah 2 - Fungsi satu giliran
Pesan pengguna ditambahkan ke riwayat, model dipanggil dengan seluruh riwayat, lalu jawabannya ikut disimpan.

In [ ]:
def kirim_giliran(masukan: str):
    """Satu giliran chat dengan riwayat percakapan."""
    messages.append(HumanMessage(content=masukan))

    try:
        response = model.invoke(messages)
        print(f"\nAsisten: {response.content}")

        messages.append(AIMessage(content=str(response.content)))
        print(f"Panjang percakapan: {len(messages)} pesan")

    except Exception as error:
        print(f"\nError: {error}")
        messages.pop()  # buang pesan pengguna yang gagal diproses agar riwayat tetap rapi

## Langkah 3 - Giliran pertama: melaporkan catatan

In [ ]:
masukan = "Shift pagi: separator V-SR1-101 level controller hunting, ESP SR-020 stabil."
print(f"\nAnda: {masukan}")
kirim_giliran(masukan)


Anda: Shift pagi: separator V-SR1-101 level controller hunting, ESP SR-020 stabil.

Asisten: Informasi shift pagi:
- V-SR1-101: level controller hunting.
- ESP SR-020: stabil.
Panjang percakapan: 3 pesan


## Langkah 4 - Giliran kedua: menanyakan ingatan
Asisten bisa menjawab karena seluruh riwayat dikirim ulang.

In [ ]:
masukan = "Apa saja yang tadi saya laporkan?"
print(f"\nAnda: {masukan}")
kirim_giliran(masukan)

print(f"\nSampai jumpa! Percakapan kita berisi {len(messages)} pesan.")


Anda: Apa saja yang tadi saya laporkan?

Asisten: Yang Anda laporkan tadi:
1. Separator V-SR1-101 level controller hunting.
2. ESP SR-020 stabil.
Panjang percakapan: 5 pesan

Sampai jumpa! Percakapan kita berisi 5 pesan.


## Langkah 5 - Mencoba sendiri (opsional)
Ubah `INTERAKTIF` menjadi `True`, jalankan sel, lalu ketik catatan atau pertanyaan. Ketik `keluar` untuk berhenti.
Pertanyaan kosong dilewati.

In [ ]:
INTERAKTIF = False  # ubah ke True untuk mencoba sendiri

if INTERAKTIF:
    while True:
        try:
            masukan = input("\nAnda: ").strip()
        except (EOFError, KeyboardInterrupt):
            print(f"\n\nSampai jumpa! Percakapan kita berisi {len(messages)} pesan.")
            break

        if masukan.lower() in ("keluar", "exit", "quit"):
            print(f"\nSampai jumpa! Percakapan kita berisi {len(messages)} pesan.")
            break

        if not masukan:
            continue

        kirim_giliran(masukan)

### Penjelasan Solusi Latihan 1

Model tidak menyimpan percakapan, jadi chatbot ini menyimpannya sendiri di daftar `messages` dan mengirim seluruh daftar itu pada setiap giliran. Itu satu-satunya cara asisten "ingat" laporan operator.

- `TODO(B1.1)`: daftar dimulai dengan satu `SystemMessage` yang menetapkan peran asisten. Pesan ini ikut terkirim di setiap panggilan.
- `TODO(B1.2)`: pesan pengguna masuk ke riwayat sebelum model dipanggil, supaya model melihat pertanyaan terbarunya.
- `TODO(B1.3)`: `model.invoke(messages)` menerima seluruh riwayat, bukan hanya kalimat terakhir.
- `TODO(B1.4)`: jawaban model dimasukkan ke riwayat sebagai `AIMessage`. Kalau baris ini dilupakan, giliran berikutnya hanya membawa pertanyaan pengguna dan asisten "lupa" apa yang ia jawab sendiri.

Pada keluaran di atas, panjang percakapan naik dari 3 pesan (sistem, pengguna, asisten) menjadi 5 setelah giliran kedua, dan jawaban giliran kedua menyebut kedua laporan dengan benar. Blok `except` memanggil `messages.pop()` agar pesan pengguna yang gagal diproses tidak tertinggal tanpa jawaban; tanpa itu, riwayat berisi dua pesan pengguna berturut-turut.

## Bab 1 - Solusi Latihan 2: Few-Shot Format Teacher
Jalankan sel satu per satu dengan Shift+Enter

Mengajari model format JSON khusus dari catatan work order lewat contoh, lalu memvalidasi hasilnya.

In [ ]:
# Langkah 1 - Menyiapkan model
import json
import os

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
    temperature=0,
)

## Langkah 2 - Contoh pengajaran dan prompt few-shot
Setiap contoh berisi catatan (input) dan keluaran JSON yang diinginkan.

In [ ]:
# Format JSON yang diajarkan
KUNCI_WAJIB = {"aset", "masalah", "tindakan", "downtime_jam"}

# Contoh pengajaran (input dan keluaran JSON)
contoh = [
    {
        "input": "ESP SR-020 trip OL, megger motor 0 Mohm, motor terbakar. Usul pulling ganti motor. downtime 5 jam.",
        "output": json.dumps({"aset": "SR-020", "masalah": "motor ESP terbakar", "tindakan": "pulling dan ganti motor", "downtime_jam": 5}, ensure_ascii=False),
    },
    {
        "input": "V-SR1-101 level control hunting, positioner LCV rusak, diganti. downtime 1 hari.",
        "output": json.dumps({"aset": "V-SR1-101", "masalah": "level control hunting karena positioner LCV rusak", "tindakan": "ganti positioner LCV", "downtime_jam": 24}, ensure_ascii=False),
    },
    {
        "input": "KN-033 V-belt pumping unit putus, ganti baru, selesai 45 menit.",
        "output": json.dumps({"aset": "KN-033", "masalah": "V-belt pumping unit putus", "tindakan": "ganti V-belt", "downtime_jam": 0.75}, ensure_ascii=False),
    },
]

template_contoh = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
few_shot = FewShotChatMessagePromptTemplate(example_prompt=template_contoh, examples=contoh)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Anda mengubah catatan work order menjadi satu objek JSON dengan kunci aset, masalah, tindakan, downtime_jam. "
               "Jawab hanya dengan JSON, tanpa teks lain."),
    few_shot,
    ("human", "{input}"),
])

chain = prompt | model

## Langkah 3 - Fungsi validasi
Kode memeriksa kunci dan tipe data, bukan model.

In [ ]:
def validasi(teks: str) -> dict:
    """Parse JSON dan periksa kunci serta tipe data. Raise ValueError bila tidak sesuai."""
    data = json.loads(teks)
    kurang = KUNCI_WAJIB - set(data)
    if kurang:
        raise ValueError(f"Kunci hilang: {sorted(kurang)}")
    if not isinstance(data["downtime_jam"], (int, float)):
        raise ValueError("downtime_jam harus berupa angka")
    return data

## Langkah 4 - Menguji dengan catatan baru

In [ ]:
uji = [
    "ESP ML-012 trip underload karena gas lock, restart manual setelah 2 jam, kembali normal.",
    "Flame scanner steam generator SG-KN-02 rusak, diganti. Unit berhenti dari 07.30 s/d 15.00.",
    "K-ML1-201 gasket flange bocor, diganti, downtime 6 jam.",
]

for catatan in uji:
    hasil = chain.invoke({"input": catatan}).content.strip()
    print(f"Catatan : {catatan}")
    try:
        data = validasi(hasil)
        print(f"   JSON valid : {json.dumps(data, ensure_ascii=False)}")
    except (json.JSONDecodeError, ValueError) as error:
        print(f"   TIDAK VALID: {error} | keluaran mentah: {hasil[:80]}")
    print()

Catatan : ESP ML-012 trip underload karena gas lock, restart manual setelah 2 jam, kembali normal.
   JSON valid : {"aset": "ML-012", "masalah": "trip underload karena gas lock", "tindakan": "restart manual", "downtime_jam": 2}

Catatan : Flame scanner steam generator SG-KN-02 rusak, diganti. Unit berhenti dari 07.30 s/d 15.00.
   JSON valid : {"aset": "SG-KN-02", "masalah": "flame scanner rusak", "tindakan": "ganti flame scanner", "downtime_jam": 7.5}

Catatan : K-ML1-201 gasket flange bocor, diganti, downtime 6 jam.
   JSON valid : {"aset": "K-ML1-201", "masalah": "gasket flange bocor", "tindakan": "ganti gasket flange", "downtime_jam": 6}



### Penjelasan Solusi Latihan 2

Soalnya mengajari model format JSON khusus lewat contoh, bukan lewat aturan panjang. Tiga contoh catatan work order beserta JSON-nya sudah cukup untuk menunjukkan kunci apa yang diharapkan.

- `TODO(B1.5)`: `ChatPromptTemplate` dengan pasangan pesan `human` dan `ai` menjadi cetakan satu contoh. `FewShotChatMessagePromptTemplate` mengulang cetakan itu untuk setiap isi `contoh`, lalu disisipkan di antara pesan sistem dan pertanyaan.
- `TODO(B1.6)`: operator `|` merangkai prompt dan model menjadi satu chain; `chain.invoke({"input": ...})` mengisi `{input}` dan memanggil model.
- `TODO(B1.7)`: validasi dikerjakan kode, bukan model. `json.loads` menolak teks yang bukan JSON, lalu kunci yang hilang dan tipe `downtime_jam` diperiksa satu per satu.

Contoh pengajaran memuat dua konversi satuan yang sengaja berbeda ("1 hari" menjadi 24, "45 menit" menjadi 0,75), dan itu terlihat pada keluaran: rentang jam 07.30 sampai 15.00 dijawab 7,5. `temperature=0` dipakai agar format stabil. Ketiga catatan baru lolos validasi pada eksekusi ini; kalau satu kali nanti tidak lolos, pesan `TIDAK VALID` menunjukkan kunci mana yang kurang, dan itu petunjuk untuk menambah contoh.

## Bab 1 - Solusi Latihan 3: Ekstraktor Data Suku Cadang
Jalankan sel satu per satu dengan Shift+Enter

Structured output: mengubah deskripsi bebas dalam berbagai format menjadi objek Pydantic yang tervalidasi.

In [ ]:
# Langkah 1 - Menyiapkan model
import os
from typing import Literal, Optional

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
    temperature=0,
)

## Langkah 2 - Skema dan pengekstrak
Field opsional dibiarkan kosong kalau informasinya tidak disebut dalam teks.

In [ ]:
class SparePart(BaseModel):
    """Data suku cadang dari deskripsi katalog atau pesan vendor."""
    nama: str = Field(description="Nama suku cadang")
    harga_juta_rupiah: Optional[float] = Field(default=None, description="Harga dalam juta rupiah, kosong bila tidak disebut")
    kategori: Literal["Elektrikal", "Mekanikal", "Instrumentasi", "Kimia", "Lainnya"] = Field(description="Kategori suku cadang")
    stok_tersedia: Optional[bool] = Field(default=None, description="Apakah stok tersedia, kosong bila tidak disebut")
    peringkat_vendor: Optional[float] = Field(default=None, ge=1, le=5, description="Peringkat vendor 1-5, kosong bila tidak disebut")
    spesifikasi: list[str] = Field(default_factory=list, description="Daftar spesifikasi utama")


prompt = ChatPromptTemplate.from_messages([
    ("system", "Ekstrak data suku cadang dari teks. Jika suatu informasi tidak disebut, biarkan kosong. Jangan menebak."),
    ("human", "{teks}"),
])

pengekstrak = prompt | model.with_structured_output(SparePart, method="function_calling")

## Langkah 3 - Mengekstrak dari tiga format deskripsi
Katalog resmi, pesan santai dari vendor, dan informasi yang tidak lengkap.

In [ ]:
deskripsi = [
    # Format katalog resmi
    "Modul VSD FD500 untuk ESP 75 kW, harga Rp 185 juta. Stok tersedia. Peringkat vendor 4,6/5. "
    "Spesifikasi: 380-480 VAC, proteksi overload, komunikasi Modbus.",
    # Format pesan santai dari vendor
    "Flame scanner steam generator, merk lokal, 12 juta aja. Barang ready di gudang Cilacap. "
    "Vendornya lumayan, bintang 4. Tahan sampai 1200 derajat C.",
    # Informasi tidak lengkap
    "Gasket flange 6 inch class 300 spiral wound, butuh 20 buah untuk turnaround.",
]

for teks in deskripsi:
    print(f"Teks: {teks[:90]}...")
    try:
        hasil = pengekstrak.invoke({"teks": teks})
        print(hasil.model_dump_json(indent=2))
    except Exception as error:
        print(f"Gagal diekstrak atau divalidasi: {error}")
    print()

Teks: Modul VSD FD500 untuk ESP 75 kW, harga Rp 185 juta. Stok tersedia. Peringkat vendor 4,6/5....
{
  "nama": "Modul VSD FD500 untuk ESP 75 kW",
  "harga_juta_rupiah": 185.0,
  "kategori": "Elektrikal",
  "stok_tersedia": true,
  "peringkat_vendor": 4.6,
  "spesifikasi": [
    "380-480 VAC",
    "proteksi overload",
    "komunikasi Modbus"
  ]
}

Teks: Flame scanner steam generator, merk lokal, 12 juta aja. Barang ready di gudang Cilacap. Ve...
{
  "nama": "Flame scanner steam generator",
  "harga_juta_rupiah": 12.0,
  "kategori": "Instrumentasi",
  "stok_tersedia": true,
  "peringkat_vendor": 4.0,
  "spesifikasi": [
    "Merk lokal",
    "Tahan sampai 1200 derajat C"
  ]
}

Teks: Gasket flange 6 inch class 300 spiral wound, butuh 20 buah untuk turnaround....
{
  "nama": "Gasket flange 6 inch class 300 spiral wound",
  "harga_juta_rupiah": null,
  "kategori": "Mekanikal",
  "stok_tersedia": null,
  "peringkat_vendor": null,
  "spesifikasi": []
}



### Penjelasan Solusi Latihan 3

Di sini skema Pydantic menggantikan prompt yang meminta JSON. `with_structured_output` membuat model mengisi objek `SparePart`, dan Pydantic memeriksanya.

- `TODO(B1.8)`: field yang boleh kosong ditulis `Optional[...] = None`. `kategori` memakai `Literal`, jadi model hanya bisa memilih salah satu dari lima nilai. `peringkat_vendor` dibatasi `ge=1, le=5`.
- `TODO(B1.9)`: `method="function_calling"` memakai fitur tool calling penyedia, yang lebih stabil di sini daripada mode JSON schema bawaan.

Perhatikan tiga hasil pada keluaran. Deskripsi katalog resmi terisi lengkap, termasuk "4,6/5" yang menjadi 4.6 dan tiga spesifikasi. Pesan santai vendor ("bintang 4", "12 juta aja") tetap masuk ke field yang tepat. Deskripsi ketiga tidak menyebut harga, jadi `harga_juta_rupiah` bernilai `null`; prompt "jangan menebak" bekerja karena field-nya memang boleh kosong. Kalau `harga_juta_rupiah` dibuat wajib, model terdorong mengarang angka.